## Building PyG Data Objects
The final result is a Data object that is correct in terms of:
+ Number of nodes
+ Number of edges
+ Correct mask
+ Correct label

##### **Import**

In [3]:
import sys
sys.path.insert(0, '..')

import torch
import pandas as pd
import numpy as np
from torch_geometric.data import Data

from src.config import (
    FEATURES_CSV_PATH, CLASSES_CSV_PATH,
    EDGELIST_CSV_PATH, TIME_SPLITS
)

##### **Load `txId` and `time_step` feature**

In [4]:
print("Loading features (txId + time_step only)...")
df_meta = pd.read_csv(
    FEATURES_CSV_PATH,
    header=None,
    usecols=[0, 1],
    names=['txId', 'time_step']
)
print(f"Total nodes: {len(df_meta):,}")
print(df_meta.head())

Loading features (txId + time_step only)...
Total nodes: 203,769
        txId  time_step
0  230425980          1
1    5530458          1
2  232022460          1
3  232438397          1
4  230460314          1


##### **Load all features (165 columns)**

In [5]:
print("Loading full features...")
all_cols = ['txId', 'time_step'] + [f'local_{i}' for i in range(93)] + [f'agg_{i}' for i in range(72)]
df_features = pd.read_csv(FEATURES_CSV_PATH, header=None, names=all_cols)
df_features['txId'] = df_features['txId'].astype('int64')
print(f"Shape: {df_features.shape}")

Loading full features...
Shape: (203769, 167)


##### **Load classes**

In [6]:
df_classes = pd.read_csv(CLASSES_CSV_PATH)
df_classes['txId'] = df_classes['txId'].astype('int64')
df_classes['class'] = df_classes['class'].astype(str)

print(df_classes['class'].value_counts())

class
unknown    157205
2           42019
1            4545
Name: count, dtype: int64


##### **Create index mapping nodes**
PyG needs index nodes from 0 to N-1. The txId in Elliptic is a random number (~7-8 digits).   
A mapping table needs to be created: txId → index.

In [7]:
all_tx_ids = df_features['txId'].values
tx_to_idx = {tx: idx for idx, tx in enumerate(all_tx_ids)}

print(f"Number of nodes: {len(tx_to_idx):,}")
print(f"Sample mapping: txId {all_tx_ids[0]} → index 0")

Number of nodes: 203,769
Sample mapping: txId 230425980 → index 0


##### **Create a node feature matrix**

In [ ]:
feature_cols = [c for c in df_features.columns if c not in ['txId', 'time_step']]
x = torch.tensor(df_features[feature_cols].values, dtype=torch.float)
print(f"x.shape: {x.shape}")